# Experimentos con PyCaret y MLflow

En este cuaderno buscamos el mejor modelo para predecir el **valor medio de una vivienda** (`median_house_value`) en California. Lo hacemos en siete pasos:

| Paso | Qué hace | Qué deja guardado |
|------|----------|-------------------|
| 1 | Parte los datos en entrenamiento y prueba | `data/train.csv` y `data/test.csv` |
| 2 | Carga las librerías y la configuración | — |
| 3 | Convierte las columnas a números | — |
| 4 | Prueba 21 modelos de regresión con PyCaret | Tabla comparativa |
| 5 | Guarda una corrida de MLflow por cada modelo y abre MLflow | Corridas en `mlflow.db` |
| 6 | Evalúa el mejor modelo con los datos de prueba | Métricas finales |
| 7 | Exporta el modelo para la API | `models/pycaret_housing.pkl` |

**Cómo ejecutarlo** (desde la raíz del repositorio):

```bash
uv sync --locked --group pycaret
uv run --group pycaret jupyter notebook pycaret_experimentos.ipynb
```

**Cómo ver las corridas:** el paso 5 abre MLflow automáticamente en http://localhost:5001. También se puede abrir desde una terminal:

```bash
uv run mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5001
```

> ⚠️ En macOS **no uses el puerto 5000** (el que MLflow usa por defecto): lo ocupa el Receptor AirPlay. En `localhost:5000` responde AirPlay y no MLflow, y por eso no aparecen los experimentos.

## Paso 1. Partir los datos y guardarlos

Esto es lo primero que hacemos, antes de entrenar nada. Separamos las 20.640 viviendas en dos grupos:

```
1553768847-housing.csv (20.640 filas)
├── data/train.csv  80 % = 16.512 filas → los modelos aprenden y se comparan con estas
└── data/test.csv   20 % =  4.128 filas → se guardan aparte y no se tocan hasta el paso 6
```

- **Entrenamiento:** con estos datos los modelos aprenden y se comparan.
- **Prueba:** como el modelo nunca los vio, nos dicen qué tan bien funcionará con viviendas nuevas.

`train_test_split` mezcla las filas al azar antes de separarlas. Con `random_state=42` la mezcla es siempre la misma: cada vez que se ejecuta el cuaderno salen las mismas filas en cada archivo. Es la misma semilla que usa `train.py`, así que los dos usan la misma partición.

Guardar los grupos en CSV deja la partición fija y a la vista: cualquier persona o script puede abrir `data/test.csv` y usar exactamente los mismos datos de prueba.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = Path('1553768847-housing.csv')
TRAIN_PATH = Path('data/train.csv')
TEST_PATH = Path('data/test.csv')
TARGET = 'median_house_value'  # la columna que queremos predecir
SEED = 42                      # misma semilla = misma partición en cada ejecución

raw = pd.read_csv(DATA_PATH)
print(f'Datos originales: {raw.shape[0]:,} filas y {raw.shape[1]} columnas')

# test_size=0.2 → 20 % para prueba; el 80 % restante es entrenamiento.
train_raw, test_raw = train_test_split(raw, test_size=0.2, random_state=SEED)

# index=False evita guardar el número de fila como una columna más.
TRAIN_PATH.parent.mkdir(exist_ok=True)
train_raw.to_csv(TRAIN_PATH, index=False)
test_raw.to_csv(TEST_PATH, index=False)
print(f'Entrenamiento: {len(train_raw):,} filas -> {TRAIN_PATH}')
print(f'Prueba:        {len(test_raw):,} filas -> {TEST_PATH}')

Datos originales: 20,640 filas y 10 columnas
Entrenamiento: 16,512 filas -> data/train.csv
Prueba:        4,128 filas -> data/test.csv


### ¿Los dos grupos se parecen?

Si la partición es buena, el precio de las viviendas tiene una distribución parecida en ambos grupos. Si el grupo de prueba tuviera, por ejemplo, solo casas caras, el resultado del paso 6 no sería representativo.

In [2]:
pd.DataFrame({
    'entrenamiento': train_raw[TARGET].describe(),
    'prueba': test_raw[TARGET].describe(),
}).round(0)

,entrenamiento,prueba
count,16512.0,4128.0
mean,207195.0,205500.0
std,115623.0,114487.0
min,14999.0,14999.0
25%,119800.0,119275.0
50%,179850.0,178650.0
75%,265125.0,263000.0
max,500001.0,500001.0


## Paso 2. Librerías y configuración

- **PyCaret** entrena y compara los modelos.
- **MLflow** guarda las corridas (configuración y resultados de cada modelo).
- **`prepare_pycaret_features`** es nuestra función para preparar las columnas; está en `pycaret_features.py` porque la API también la usa.

In [3]:
import os
import subprocess
import sys
import time
import urllib.request

# MLflow imprime muchos mensajes en cada corrida; así solo muestra errores.
# Debe definirse antes de importar mlflow.
os.environ.setdefault('MLFLOW_LOGGING_LEVEL', 'ERROR')
os.environ.setdefault('MLFLOW_DISABLE_AGENT_HINT', '1')

import joblib
import mlflow
from pycaret.regression import RegressionExperiment
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from pycaret_features import prepare_pycaret_features

MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

## Paso 3. Preparar las columnas para el modelo

A partir de aquí leemos los datos **desde los archivos guardados**, no desde la memoria. Así comprobamos que los archivos sirven por sí solos.

Los modelos solo entienden números, por eso `prepare_pycaret_features` hace dos cambios:

1. `ocean_proximity` es texto (`"INLAND"`, `"NEAR BAY"`…) y se cambia por un código fijo (`INLAND` → 1, `NEAR BAY` → 3…).
2. A algunas viviendas les falta `total_bedrooms`. Ese vacío se rellena con `-1` y se agrega la columna `bedrooms_missing` (1 = faltaba el dato), para que el modelo sepa que ese `-1` no es un conteo real.

La API usa esta misma función. Si el cuaderno y la API transformaran los datos de forma distinta, el modelo recibiría en producción algo diferente a lo que aprendió. Este es su código:

In [4]:
import inspect

import pycaret_features

print(inspect.getsource(pycaret_features))

"""Adaptación determinista del CSV para PyCaret y la API de inferencia."""

import pandas as pd


# Fijar los códigos evita depender del orden de categorías de cada lote.
OCEAN_CODES = {
    "<1H OCEAN": 0,
    "INLAND": 1,
    "ISLAND": 2,
    "NEAR BAY": 3,
    "NEAR OCEAN": 4,
}

# Este orden coincide con las columnas utilizadas al entrenar el pipeline.
PYCARET_FEATURES = [
    "longitude",
    "latitude",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income",
    "ocean_proximity_code",
    "bedrooms_missing",
]


def prepare_pycaret_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Convierte entradas originales en columnas numéricas de esquema fijo.

    El valor -1 representa dormitorios desconocidos; la columna indicadora
    permite distinguirlo de un conteo observado. No se calculan estadísticas
    con el lote de inferencia, así que el resultado es igual en train y serve.
    """
    # Copiar los datos e

In [5]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

# X = columnas de entrada; y = valor que queremos predecir.
X_train = prepare_pycaret_features(train)
y_train = train[TARGET]
X_test = prepare_pycaret_features(test)
y_test = test[TARGET]

X_train.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity_code,bedrooms_missing
0,-117.03,32.71,33,3126,627.0,2300,623,3.2596,4,0
1,-118.16,33.77,49,3382,787.0,1314,756,3.8125,4,0
2,-120.48,34.66,4,1897,331.0,915,336,4.1563,4,0
3,-117.11,32.69,36,1421,367.0,1418,355,1.9425,4,0
4,-119.80,36.78,43,2382,431.0,874,380,3.5542,1,0


## Paso 4. Probar los modelos con PyCaret

PyCaret entrena y compara muchos modelos con pocas líneas de código. Probamos **21**, agrupados por familia:

| Familia | Modelos |
|---------|---------|
| Referencia | `dummy` (siempre predice el promedio; cualquier modelo útil debe superarlo) |
| Lineales | `lr` regresión lineal, `ridge`, `lasso`, `en` Elastic Net, `lar` y `llar` (Least Angle), `omp` (Orthogonal Matching Pursuit), `br` Bayesian Ridge, `ard` |
| Lineales robustos a valores atípicos | `huber`, `ransac`, `par` (Passive Aggressive) |
| Vecinos y kernels | `knn` K vecinos, `svm` máquina de vectores de soporte |
| Árboles | `dt` árbol de decisión |
| Ensambles de árboles | `rf` Random Forest, `et` Extra Trees, `gbr` Gradient Boosting, `ada` AdaBoost |
| Redes neuronales | `mlp` perceptrón multicapa |

Del catálogo de PyCaret quedan fuera `kr` (Kernel Ridge) y `tr` (TheilSen): Kernel Ridge arma una matriz de 13.209 × 13.209 por cada fold (~1,4 GB) y TheilSen tarda demasiado con este volumen de datos.

**`normalize=True`** pone todas las columnas en la misma escala antes de entrenar. Los árboles no lo necesitan, pero `knn`, `svm` y `mlp` miden distancias o ajustan pesos y funcionan mal si una columna va de 0 a 15 y otra de 0 a 40.000. El escalado queda dentro del pipeline exportado, así que la API no tiene que hacer nada extra.

### Cómo parte PyCaret los datos de entrenamiento

Dentro de `data/train.csv`, PyCaret vuelve a partir los datos:

```
data/train.csv (16.512 filas)
├── 80 % = 13.209 filas → validación cruzada de 5 partes (fold=5)
│     ├── parte 1 ─┐
│     ├── parte 2  │  en cada ronda el modelo aprende con 4 partes
│     ├── parte 3  ├─ y se evalúa con la que queda; se hacen 5 rondas
│     ├── parte 4  │  y se promedia el resultado
│     └── parte 5 ─┘
└── 20 % =  3.303 filas → reserva interna de PyCaret (train_size=0.8)
```

La **validación cruzada** hace que la comparación no dependa de una sola división de los datos. La tabla queda ordenada por **RMSE**: el error típico de la predicción, en dólares. **Menor es mejor.** También aparece **R²**: qué tanto de la variación del precio explica el modelo (1 = perfecto, 0 = igual que predecir el promedio, negativo = peor que el promedio).

In [6]:
# PyCaret necesita las entradas y el objetivo en una sola tabla.
training_data = X_train.copy()
training_data[TARGET] = y_train

experiment = RegressionExperiment(
    target=TARGET,
    session_id=SEED,  # semilla de PyCaret
    train_size=0.8,   # PyCaret aparta un 20 % interno para validar al mejor modelo
    fold=5,           # validación cruzada de 5 partes
    normalize=True,   # misma escala para todas las columnas
    n_jobs=2,         # núcleos de CPU que puede usar
).fit(training_data)

# Comprobar los tamaños del dibujo anterior con los datos reales.
print(f'Validación cruzada: {len(experiment.X_train):,} filas en {experiment.fold} partes')
print(f'Reserva interna:    {len(experiment.X_test):,} filas')

Validación cruzada: 13,209 filas en 5 partes
Reserva interna:    3,303 filas


In [7]:
MODEL_IDS = [
    'dummy',
    'lr', 'ridge', 'lasso', 'en', 'lar', 'llar', 'omp', 'br', 'ard',
    'huber', 'ransac', 'par',
    'knn', 'svm',
    'dt',
    'rf', 'et', 'gbr', 'ada',
    'mlp',
]

start = time.time()
comparison = experiment.compare_models(
    include=MODEL_IDS,
    sort='RMSE',               # ordenar del menor al mayor error
    n_select=len(MODEL_IDS),   # devolver todos los modelos, no solo el mejor
    turbo=False,               # no omitir los modelos lentos
    errors='raise',            # detenerse si un modelo falla, en vez de saltarlo
)
print(f'{len(MODEL_IDS)} modelos entrenados en {time.time() - start:.0f} segundos')

leaderboard = comparison.leaderboard
leaderboard

21 modelos entrenados en 60 segundos


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,rf,33240.6880,2.544621e+09,50432.0136,0.8080,0.2442,0.1855
1,et,35201.6981,2.768156e+09,52609.6308,0.7911,0.2520,0.1957
2,gbr,38527.9735,3.026093e+09,55005.5245,0.7717,0.2739,0.2195
3,knn,41750.1804,3.821588e+09,61815.6316,0.7117,0.2917,0.2260
4,ard,50340.1981,4.785389e+09,69162.8308,0.6390,0.4195,0.3017
5,br,50338.2410,4.786167e+09,69168.5116,0.6389,0.4169,0.3017
6,ridge,50338.4057,4.786177e+09,69168.5637,0.6389,0.4174,0.3017
7,llar,50338.9463,4.786215e+09,69168.8056,0.6389,0.4188,0.3017
8,lasso,50338.9504,4.786216e+09,69168.8077,0.6389,0.4188,0.3017
9,lr,50338.8154,4.786226e+09,69168.8721,0.6389,0.4190,0.3017


**Cómo leer la tabla:**

- Los ensambles de árboles (`rf`, `et`, `gbr`) quedan arriba porque capturan relaciones no lineales, como el efecto de la ubicación.
- `knn` queda cuarto gracias a `normalize=True`: sin escalar, la población (miles) pesaría mucho más que el ingreso (0 a 15) al medir distancias.
- Los lineales (`lr`, `ridge`, `lasso`, `br`…) quedan casi empatados: todos ajustan la misma recta con pequeñas variaciones.
- `dummy` marca la línea base. Los modelos con R² negativo lo hacen peor que predecir el promedio: `svm` y `mlp`, con su configuración por defecto, no se adaptan a precios de cientos de miles de dólares (necesitarían ajustar hiperparámetros con `tune_model`), y `lar` es inestable cuando hay columnas muy correlacionadas, como `total_rooms`, `total_bedrooms` y `households`.

## Paso 5. Guardar las corridas en MLflow

Una **corrida** (*run*) de MLflow es el registro de un entrenamiento. Guarda:

- **Parámetros**: cómo se configuró el modelo (por ejemplo, cuántos árboles tiene).
- **Métricas**: qué resultados obtuvo (RMSE, MAE, R²…).

Creamos una corrida por cada modelo, todas dentro del experimento `California_Housing_PyCaret`. PyCaret 4 no las guarda solo, por eso lo hacemos con un ciclo `for`.

Las corridas quedan en el archivo `mlflow.db`. Si existe la variable de entorno `MLFLOW_TRACKING_URI` (como en Docker), se envían a ese servidor.

In [8]:
EXPERIMENT_NAME = 'California_Housing_PyCaret'
# Docker define MLFLOW_TRACKING_URI con su servidor; si no existe, se usa el archivo local.
# Se lee antes de set_tracking_uri, porque MLflow también escribe esa variable.
remote_server = os.getenv('MLFLOW_TRACKING_URI')
tracking_uri = remote_server or 'sqlite:///mlflow.db'
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment(EXPERIMENT_NAME)

# ranked_ids, models y las filas de leaderboard están en el mismo orden (del mejor al peor).
for position, model_id in enumerate(comparison.ranked_ids):
    pipeline = comparison.models[position]
    row = leaderboard.iloc[position]

    with mlflow.start_run(run_name=f'candidate_{model_id}'):
        # Parámetros: el código del modelo y su configuración.
        # pipeline[-1] es el modelo (el último paso, después del preprocesamiento).
        mlflow.log_param('model_id', model_id)
        mlflow.log_param('cv_folds', 5)
        mlflow.log_params(pipeline[-1].get_params())

        # Métricas: promedios de la validación cruzada.
        for metric in ['MAE', 'MSE', 'RMSE', 'R2', 'RMSLE', 'MAPE']:
            mlflow.log_metric(f'cv_{metric.lower()}', float(row[metric]))

print(f'{len(comparison.ranked_ids)} corridas guardadas en {tracking_uri}')

21 corridas guardadas en sqlite:///mlflow.db


### Revisar las corridas guardadas

Leemos de vuelta las corridas desde MLflow para comprobar que se guardaron. Cada vez que se ejecuta el cuaderno se agregan corridas nuevas; aquí mostramos solo las de esta ejecución.

In [9]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    filter_string="tags.mlflow.runName LIKE 'candidate_%'",
    order_by=['attributes.start_time DESC'],
    max_results=len(MODEL_IDS),
)

runs_table = runs[['tags.mlflow.runName', 'metrics.cv_rmse', 'metrics.cv_mae', 'metrics.cv_r2']]
runs_table.columns = ['corrida', 'cv_rmse', 'cv_mae', 'cv_r2']
runs_table.sort_values('cv_rmse').reset_index(drop=True)

,corrida,cv_rmse,cv_mae,cv_r2
0,candidate_rf,50432.0136,33240.6880,0.8080
1,candidate_et,52609.6308,35201.6981,0.7911
2,candidate_gbr,55005.5245,38527.9735,0.7717
3,candidate_knn,61815.6316,41750.1804,0.7117
4,candidate_ard,69162.8308,50340.1981,0.6390
5,candidate_br,69168.5116,50338.2410,0.6389
6,candidate_ridge,69168.5637,50338.4057,0.6389
7,candidate_llar,69168.8056,50338.9463,0.6389
8,candidate_lasso,69168.8077,50338.9504,0.6389
9,candidate_lr,69168.8721,50338.8154,0.6389


### Abrir MLflow para ver los experimentos

La siguiente celda arranca la interfaz web de MLflow en segundo plano, leyendo el mismo `mlflow.db`, y espera a que responda. Usa el puerto **5001** porque en macOS el 5000 lo ocupa AirPlay.

En la interfaz:

1. Entra al experimento **California_Housing_PyCaret** (columna izquierda).
2. Verás una fila por corrida (`candidate_rf`, `candidate_gbr`…). Haz clic en una para ver sus parámetros y métricas.
3. Para comparar, marca varias corridas y pulsa **Compare**.

En Docker no hace falta: el servicio `mlflow` ya está en http://localhost:5050.

In [10]:
MLFLOW_PORT = 5001
mlflow_url = f'http://localhost:{MLFLOW_PORT}'


def mlflow_is_running(url):
    try:
        return urllib.request.urlopen(f'{url}/health', timeout=1).status == 200
    except OSError:
        return False


if remote_server:
    # En Docker, el servidor de MLflow es otro contenedor.
    print('MLflow corre en el servicio de Docker: abrir http://localhost:5050')
else:
    if not mlflow_is_running(mlflow_url):
        # sys.executable es el Python de este cuaderno, que ya tiene MLflow instalado.
        mlflow_ui = subprocess.Popen(
            [sys.executable, '-m', 'mlflow', 'ui',
             '--backend-store-uri', tracking_uri, '--port', str(MLFLOW_PORT)],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        # El servidor tarda unos segundos en arrancar.
        for _ in range(60):
            if mlflow_is_running(mlflow_url):
                break
            time.sleep(1)

    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
    print(f'MLflow está corriendo: {mlflow_url}')
    print(f'Experimento {EXPERIMENT_NAME}: {mlflow_url}/#/experiments/{experiment_id}')
    print('Para detenerlo: reiniciar el kernel o ejecutar mlflow_ui.terminate()')

MLflow está corriendo: http://localhost:5001
Experimento California_Housing_PyCaret: http://localhost:5001/#/experiments/3
Para detenerlo: reiniciar el kernel o ejecutar mlflow_ui.terminate()


## Paso 6. Evaluar el mejor modelo con los datos de prueba

Ahora sí usamos `data/test.csv`. El mejor modelo se eligió **sin mirar esos datos**, así que este resultado es una estimación honesta de cómo funcionará con viviendas nuevas.

Si el RMSE de prueba es parecido al RMSE de la validación cruzada, el modelo generaliza bien. Si fuera mucho mayor, estaría memorizando los datos de entrenamiento (*sobreajuste*).

In [11]:
best_id = comparison.ranked_ids[0]
predictions = comparison.best.predict(X_test)

test_metrics = {
    'test_rmse': float(mean_squared_error(y_test, predictions) ** 0.5),
    'test_mae': float(mean_absolute_error(y_test, predictions)),
    'test_r2': float(r2_score(y_test, predictions)),
}

print(f'Mejor modelo: {best_id}')
print(f"RMSE en validación cruzada: {leaderboard.iloc[0]['RMSE']:,.0f}")
print(f"RMSE en prueba:             {test_metrics['test_rmse']:,.0f}")
print(f"MAE en prueba:              {test_metrics['test_mae']:,.0f}")
print(f"R² en prueba:               {test_metrics['test_r2']:.4f}")

Mejor modelo: rf
RMSE en validación cruzada: 50,432
RMSE en prueba:             51,351
MAE en prueba:              33,148
R² en prueba:               0.7988


## Paso 7. Exportar el modelo y guardar la corrida final

1. **`finalize_model`** vuelve a entrenar el mejor modelo con **todos** los datos de entrenamiento (incluida la reserva interna de PyCaret). Más datos suelen dar un modelo un poco mejor.
2. **`save_model`** guarda el *pipeline* completo (escalado + modelo) en `models/pycaret_housing.pkl`. Ese es el archivo que carga la API.
3. Una última corrida, `selected_final_model`, guarda en MLflow las métricas de prueba, la tabla comparativa y el archivo del modelo. También aparece en la interfaz de MLflow.

In [12]:
final_pipeline = experiment.finalize_model(comparison.best).pipeline

# save_model agrega la extensión .pkl al nombre.
experiment.save_model(final_pipeline, str(MODEL_DIR / 'pycaret_housing'))
model_path = MODEL_DIR / 'pycaret_housing.pkl'

leaderboard_path = MODEL_DIR / 'pycaret_leaderboard.csv'
leaderboard.to_csv(leaderboard_path, index=False)

with mlflow.start_run(run_name='selected_final_model'):
    mlflow.log_param('selected_model_id', best_id)
    mlflow.log_metrics(test_metrics)
    mlflow.log_artifact(str(leaderboard_path))
    mlflow.log_artifact(str(model_path))

print(f'Modelo exportado: {model_path}')

Modelo exportado: models/pycaret_housing.pkl


### Comprobar el modelo exportado

Cargamos el archivo `.pkl` igual que lo hace la API (`pycaret_app.py`) y predecimos tres viviendas de prueba. Si funciona aquí, funcionará en la API.

In [13]:
served_model = joblib.load(model_path)

sample = test.head(3)
pd.DataFrame({
    'valor_real': sample[TARGET],
    'prediccion': served_model.predict(prepare_pycaret_features(sample)).round(0),
})

,valor_real,prediccion
0,47700,60221.0
1,45800,181413.0
2,500001,444258.0


## Servir el modelo

La API está en `pycaret_app.py`. Carga el modelo una sola vez al arrancar, así que después de exportar un modelo nuevo hay que reiniciarla.

```bash
# Local:
uv run uvicorn pycaret_app:app --reload
# Abrir http://localhost:8000/docs para probar /predict

# Con Docker Compose (MLflow + notebook + API):
docker compose up --build
# Ejecutar este cuaderno en http://localhost:8888 (el token aparece en los logs)
docker compose restart api
# Ver las corridas en http://localhost:5050
```

La API responde `503` mientras no exista `models/pycaret_housing.pkl`.